# 13. Lecke - Ügynök memória Cognee Tudásgráfokkal


## Beállítás

Ez a jegyzetfüzet bemutatja, hogyan lehet egy intelligens **kódtámogatót** építeni tartós memóriával a [**Cognee**](https://www.cognee.ai/) tudásgráfok és a **Microsoft Agent Framework** (MAF) segítségével.

A Cognee az strukturálatlan szöveget strukturált, lekérdezhető tudásgráffá alakítja, amelyet vektorembeddingek támogatnak — így az ügynököd gazdag, kapcsolatokkal átfogott hosszú távú memóriával rendelkezik.

### Amit megtanulsz
1. **Tudásgráfok építése**: Fejlesztői profilok és bevált gyakorlatok átalakítása strukturált, lekérdezhető tudássá.
2. **Cognee integrálása MAF-pal**: `@tool` függvények használata, hogy egy MAF ügynök lekérdezhesse a Cognee tudásgráfját.
3. **Munkamenethez kötött beszélgetések**: Kontextus megőrzése több kérdés során ugyanabban a munkamenetben.
4. **Hosszú távú memória**: Fontos tudás megőrzése a munkamenetek között és előhívása új beszélgetésekben.

### Előfeltételek
- Python 3.9+
- Lokálisan futó Redis (`docker run -d -p 6379:6379 redis`) a munkamenet-kezeléshez
- LLM API kulcs (pl. OpenAI) — állítsd be a `LLM_API_KEY` értéket a `.env` fájlban
- `CACHING=true` a `.env` fájlban (szükséges a Cognee munkamenetekhez)
- Azure AI Foundry projekt egy telepített chat modellel
- `AZURE_AI_PROJECT_ENDPOINT` és `AZURE_AI_MODEL_DEPLOYMENT_NAME` a `.env` fájlban
- Azure CLI hitelesítve (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Ügynök memóriatípusai

Ez a jegyzetfüzet ugyanazokat a három memóriatípust vizsgálja, mint a fő 13. lecke jegyzetfüzete, de a Cognee-t használja hosszú távú memória háttérként:

| Memóriatípus | Mechanizmus | Élettartam |
|---|---|---|
| **Munkamemória** | `agent.create_session()` (MAF) | Egyetlen beszélgetés |
| **Rövid távú** | Cognee munkamenet gyorsítótár (Redis) | Egyetlen munkamenet |
| **Hosszú távú** | Cognee tudásgráf + vektorok | Állandó |

### A Cognee memóriaarchitektúrája
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Készítse elő a Cognee Tárolót


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 1. rész — A tudásbázis felépítése

Háromféle adatot dolgozunk fel, hogy átfogó tudásbázist hozzunk létre kódoló asszisztensünk számára:

1. **Fejlesztői profil** — személyes szakértelem és műszaki háttér  
2. **Python legjobb gyakorlatai** — a Python Zene gyakorlati iránymutatásokkal  
3. **Korábbi beszélgetések** — múltbeli kérdés-válasz alkalmak fejlesztők és AI asszisztensek között


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## A tudásgráf megjelenítése

A Cognee képes interaktív HTML megjelenítést készíteni a kinyert entitásokról és kapcsolatokról.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memória gazdagítása a Memify segítségével

A `memify()` elemzi a tudásgráfot, és intelligens szabályokat generál — felismerve a mintákat, bevált gyakorlatokat és a fogalmak közötti kapcsolatokat.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — MAF ügynök a Cognee eszközökkel

Most létrehozunk egy MAF ügynököt, amely a `@tool` függvényeken keresztül képes lekérdezni a Cognee tudásgráfját. Ez lehetővé teszi az ügynök számára, hogy kihasználja a gráf-tudatos szemantikus keresés teljes erejét, miközben a párbeszédkörnyezetet munkameneteken keresztül megtartja.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Munkamemória munkamenetekkel

Az `AgentSession` (az `agent.create_session()` segítségével létrehozva) munkamemóriát biztosít egy munkameneten belül. Az agent visszautalhat korábbi üzenetekre, miközben lekérdezi a Cognee hosszú távú tudásgráfját is.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Új munkamenet — A hosszú távú memória megmarad

Egy új munkamenet indítása törli a munkamemóriát, de a Cognee tudásgráf továbbra is elérhető. Az ügynök ugyanazt a hosszú távú tudást képes lekérni egy teljesen új beszélgetés során.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Összefoglaló

Ebben a jegyzetfüzetben egy olyan kódoló asszisztenst építettél, amely ötvözi a **MAF munkamemóriáját** (`agent.create_session()`) a **Cognee hosszú távú tudásgráfjával**.

### Amit megtanultál
1. **Tudásgráf építése**: A Cognee feldolgozza a strukturálatlan szöveget, és gráfot + vektormemóriát épít.
2. **Gráf gazdagítása memify-jal**: Leszármaztatott tények és gazdagabb kapcsolatok létrehozása a meglévő gráfra.
3. **MAF + Cognee integráció**: A `@tool` funkciók lehetővé teszik, hogy egy MAF ügynök természetesen lekérdezze a Cognee gráfját.
4. **Munkamemória + hosszú távú memória**: Az `AgentSession` (az `agent.create_session()` segítségével) biztosít munkamenet-környezetet, míg a Cognee tartós tudást nyújt.
5. **Szűrt keresés NodeSets-szel**: A tudásgráf adott részhalmazainak célozása (pl. csak alapelvek).

### Fő tanulságok
- A **Cognee** a nyers szöveget strukturált, kapcsolatokkal rendelkező memóriává alakítja — erősebb, mint egy lapos vektortár.
- A **`@tool` funkciók** tiszta hidat képeznek a MAF ügynökök és külső tudásrendszerek között.
- Az **`AgentSession`** (az `agent.create_session()` segítségével) az egyes beszélgetések kontextusát szétválasztja a hosszú távú tudástól.
- Ugyanaz a tudásgráf több munkamenetet és ügynököt szolgál ki.

### Valós alkalmazások
- **Fejlesztői copilotok**: Kódellenőrzés, incidenselemzés, architektúra asszisztensek
- **Ügyfélközpontú copilotok**: Támogató ügynökök termékdokumentációk, GYIK és CRM jegyzetek fölött
- **Belső szakértői copilotok**: Szabályzat, jogi vagy biztonsági asszisztensek irányelvek alapján
- **Egységes adatrétegek**: Strukturált és nem strukturált adatok egyetlen lekérdezhető gráfba kombinálása

### Következő lépések
- Kísérletezés időbeli tudatossággal a Cognee-ben
- Domain-specifikus gráfminőséghez OWL ontológia definiálása
- Felhasználói visszacsatolások beépítése a visszakeresés folyamatos javításához
- Többügynökös rendszerekre skálázás, amelyek ugyanazt a Cognee memóriaszintet használják


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:  
Ez a dokumentum az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár igyekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti dokumentum az anyanyelvén tekintendő tekintélyes forrásnak. Fontos információk esetén szakmai emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő esetleges félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
